In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight

# 1. Load your dataset
file_path = 'TRAINING-DATA-WATER.xlsx'
df = pd.read_excel(file_path, sheet_name=0)

print(f"Original dataset loaded! Total rows: {df.shape[0]}")
print(f"Unique coordinates: {df[['Latitude', 'Longitude']].drop_duplicates().shape[0]}")

# 2. Spatial blocks: points closer than 1 km (chained) belong to the same block,
#    so a whole block is either in train or in test (no spatial leakage)
BLOCK_M = 1000
xy = np.c_[df['Latitude'] * 111320,
           df['Longitude'] * 111320 * np.cos(np.radians(df['Latitude'].mean()))]
blocks = pd.Series(fcluster(linkage(xy, 'single'), BLOCK_M, 'distance'), index=df.index)
print(f"Spatial blocks: {blocks.nunique()}")

# 3. Encode Categorical Features & Target (coordinates are NOT used as predictors)
target_encoder = LabelEncoder()
y = pd.Series(target_encoder.fit_transform(df['Safety_class'].str.strip().str.title()), index=df.index)
print("Classes:", dict(zip(target_encoder.classes_, range(len(target_encoder.classes_)))))

X = pd.get_dummies(df.drop(columns=['ID', 'Latitude', 'Longitude', 'Safety_class']),
                   columns=['Land_use', 'Water_type'], dtype=int)

# 4. Spatial Stratified Group 5-Fold Cross-Validation Setup for robust evaluation
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []

# Class order: [Safe, Unsafe]. A small boost to Unsafe so unsafe sites are not missed
cost_weights = np.array([1.0, 1.5])

print("\nRunning Spatial Stratified 5-Fold Cross-Validation for Logistic Regression...")
for train_idx, test_idx in sgkf.split(X, y, blocks):
    X_train_raw, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train_raw, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Upsample training split only to ensure robust learning for the minority class
    train_df = X_train_raw.copy()
    train_df['target'] = y_train_raw
    class_dfs = [train_df[train_df['target'] == i] for i in range(2)]
    expanded_train_dfs = []

    target_counts = {0: 300, 1: 300}
    for i, cls_df in enumerate(class_dfs):
        if len(cls_df) > 0:
            resampled_cls = resample(cls_df, replace=True, n_samples=target_counts[i], random_state=42 + i)
            expanded_train_dfs.append(resampled_cls)
        else:
            dummy_row = train_df.iloc[0:1].copy()
            dummy_row['target'] = i
            resampled_cls = resample(dummy_row, replace=True, n_samples=target_counts[i], random_state=42 + i)
            expanded_train_dfs.append(resampled_cls)

    train_final = pd.concat(expanded_train_dfs).reset_index(drop=True)
    X_train = train_final.drop(columns=['target'])
    y_train = train_final['target'].values

    # Logistic Regression pipeline with feature scaling
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(
            solver='lbfgs',
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ))
    ])
    model.fit(X_train, y_train)

    # Predict probabilities and apply cost weights
    probs = model.predict_proba(X_test)
    calibrated_probs = probs * cost_weights
    y_pred = np.argmax(calibrated_probs, axis=1)

    all_y_true.extend(y_test.values)
    all_y_pred.extend(y_pred)

# 5. Evaluate Cross-Validation Results
acc = accuracy_score(all_y_true, all_y_pred)
print(f"\n================ Logistic Regression Spatial Cross-Validated Results ================")
print(f"Overall Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=target_encoder.classes_, zero_division=0))
print("Confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(all_y_true, all_y_pred))

# 6. Train Final Production Model on Full Data and Save to Disk
print("\nTraining final production Logistic Regression model on full dataset...")
full_train_df = X.copy()
full_train_df['target'] = y
full_class_dfs = [full_train_df[full_train_df['target'] == i] for i in range(2)]
full_expanded_dfs = []

for i, cls_df in enumerate(full_class_dfs):
    resampled_cls = resample(cls_df, replace=True, n_samples=350, random_state=42 + i)
    full_expanded_dfs.append(resampled_cls)

final_train_full = pd.concat(full_expanded_dfs).reset_index(drop=True)
X_train_final = final_train_full.drop(columns=['target'])
y_train_final = final_train_full['target'].values

final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])
final_model.fit(X_train_final, y_train_final)

# Save model to 'models/' directory
os.makedirs('models', exist_ok=True)
joblib.dump(final_model, 'models/water_contamination_logreg_model.pkl')

# Save the column order and class names (needed to encode new data for the map)
joblib.dump({'features': list(X.columns), 'classes': list(target_encoder.classes_)},
            'models/water_contamination_features.pkl')

print("Logistic Regression production model successfully saved as 'models/water_contamination_logreg_model.pkl'!")
